# XLM-R with Katip's recipe · is a multilingual base better than BERTurk?

**What this does.** Fine-tunes `FacebookAI/xlm-roberta-base` (Meta AI, MIT) exactly like Katip Mask Base v2 (Katip's synthetic
documents + 17,967 real Wiki NER sentences, 3 epochs) so the only difference is the base model. On your Mac this run
ran out of memory; a T4 has 15 GB of its own.

| | |
|---|---|
| Time | ~5 min smoke test, then ~45-75 min on a free T4 |
| Output | `katip-xlmr.zip` (ONNX int8 ~280 MB + PyTorch weights) and the training scores |
| Question it answers | does XLM-R beat BERTurk (Base v2: Wikipedia recall ~90%) enough to justify 2.5x the size? |

**How to run:** Runtime > Run all.

In [ ]:
# 1 · Check the GPU, get the code. Run this cell first. (Takes ~1 min.)
import os, subprocess, sys
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE. In Colab: Runtime > Change runtime type > T4 GPU. In Kaggle: Settings > Accelerator > GPU T4.")
if not os.path.exists("katip"):
    subprocess.run("git clone -q --depth 1 -b legal-study https://github.com/fikrifirat/katip", shell=True, check=True)
os.chdir("katip")
subprocess.run(f"{sys.executable} -m pip install -q regex tokenizers onnx onnxruntime faker huggingface_hub 'transformers>=4.44' accelerate", shell=True, check=True)
print(subprocess.run("git log --oneline -1", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Optional · Hugging Face token, only needed to upload at the end.
# Colab: the key icon on the left > add a secret named HF_TOKEN (write access) and allow this notebook.
# Kaggle: Add-ons > Secrets > HF_TOKEN. Nothing is hard-coded here.
HF_TOKEN = None
try:
    from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
print("HF token:", "found" if HF_TOKEN else "none (fine: you can download the zip instead)")

In [ ]:
!pip install -q sentencepiece protobuf

## 2 · Smoke test (3-5 min)

In [ ]:
!cd ml/berturk && python train_berturk.py --base FacebookAI/xlm-roberta-base --real-text --limit 300 --epochs 1 --out out_smoke 2>&1 | tail -8

## 3 · Full run (~1 h)

In [ ]:
!cd ml/berturk && python train_berturk.py --base FacebookAI/xlm-roberta-base --real-text --out out_xlmr --bs 16 2>&1 | tail -20

In [ ]:
import json, glob
for f in glob.glob("ml/berturk/out_xlmr/*.json"):
    if "result" in f or "train" in f: print(f); print(open(f).read()[:1500])

In [ ]:
# Save the result: a zip you can download (left panel > Files) and, if you set HF_TOKEN, a PRIVATE upload to Hugging Face.
# Make it public later on the model page (Settings > Visibility) after checking the scores.
import shutil, os
z = shutil.make_archive("katip-xlmr", "zip", "ml/berturk/out_xlmr")
print("zip:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
try:
    from google.colab import files; files.download(z)
except Exception:
    pass
UPLOAD = False   # set True to upload
if UPLOAD and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo("f1k/katip-xlmr-experiment", private=True, exist_ok=True)
    api.upload_folder(folder_path="ml/berturk/out_xlmr", repo_id="f1k/katip-xlmr-experiment", commit_message="trained in a notebook (katip-xlmr)")
    print("uploaded to https://huggingface.co/f1k/katip-xlmr-experiment (private)")

**Next:** send the scores back in the Katip thread; we run it on KVKK-Bench against Base v2.